# Project 2 — Supervised Learning: Fraud Detection Pipeline

### Objective
Build and tune a classification pipeline for detecting potentially fraudulent transactions in a highly imbalanced dataset.

**Required techniques covered**
- Class-imbalance handling with **SMOTE**
- **Logistic Regression** and **Random Forest**
- Scikit-learn / imbalanced-learn pipelines
- Hyperparameter tuning
- Evaluation using **Precision, Recall, and ROC-AUC** rather than relying on Accuracy

> **Important dataset note:** The supplied Excel file contains 1,200 transactions but **does not contain a real fraud/legitimate target column**. Therefore, this project creates a clearly labelled **proxy fraud target** from transparent transaction-risk rules so the supervised-learning pipeline can still be demonstrated. This is suitable for a learning/demo project, but it must **not** be presented as a real-world fraud model or as ground-truth fraud detection. If a genuine `Fraud` label becomes available, replace the proxy-target section and retrain the same pipeline.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import joblib

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


In [ ]:
# Load the sanitized, project-ready dataset.
# The original workbook is intentionally not copied into the repository because it
# contains identifier/address/tracking fields that are not needed for modeling.

DATA_PATH = "data/fraud_detection_dataset_proxy.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 1. Data Audit

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["FraudFlag"].value_counts().rename(index={0: "Legitimate", 1: "Fraud Proxy"}).to_frame("count"))

fraud_rate = df["FraudFlag"].mean() * 100
print(f"Proxy positive rate: {fraud_rate:.2f}%")


## 2. Proxy Fraud Target

Because the original dataset has no fraud label, the proxy target is created from a transparent risk score:

- unusually high `TotalPrice` (top 10%) → +2
- `Quantity == 5` → +1
- `ItemsInCart >= 9` → +1
- `PaymentMethod` is Online or Gift Card → +1
- `OrderStatus` is Cancelled or Returned → +1
- missing `CouponCode` → +1

Transactions with a score of **4 or more** receive `FraudFlag = 1`.

The score itself is **not** used as a model feature, preventing direct target leakage from that engineered score.


In [ ]:
# Reconstruct the proxy-target logic for auditability.
# The dataset already contains FraudFlag, so this cell independently checks the rule.

check = df.copy()
q90 = check["TotalPrice"].quantile(0.90)

proxy_score = (
    (check["TotalPrice"] >= q90).astype(int) * 2
    + (check["Quantity"] == 5).astype(int)
    + (check["ItemsInCart"] >= 9).astype(int)
    + (check["PaymentMethod"].isin(["Online", "Gift Card"])).astype(int)
    + (check["OrderStatus"].isin(["Cancelled", "Returned"])).astype(int)
    + check["CouponCode"].isna().astype(int)
)

print("Proxy rule matches stored FraudFlag:", np.array_equal((proxy_score >= 4).astype(int), check["FraudFlag"]))
display(pd.Series(proxy_score, name="FraudScore_Proxy").value_counts().sort_index().to_frame())


## 3. Exploratory Analysis

In [ ]:
# Class balance
counts = df["FraudFlag"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Legitimate", "Fraud Proxy"], counts.values)
plt.title("Class Distribution")
plt.ylabel("Transactions")
plt.tight_layout()
plt.show()

# Total price by class
plt.figure(figsize=(7, 4))
plt.boxplot(
    [df.loc[df["FraudFlag"] == 0, "TotalPrice"],
     df.loc[df["FraudFlag"] == 1, "TotalPrice"]],
    labels=["Legitimate", "Fraud Proxy"]
)
plt.title("Total Price by Class")
plt.ylabel("Total Price")
plt.tight_layout()
plt.show()

# Payment method vs class
payment_ct = pd.crosstab(df["PaymentMethod"], df["FraudFlag"])
payment_ct.plot(kind="bar", figsize=(8, 4))
plt.title("Payment Method by Class")
plt.xlabel("Payment Method")
plt.ylabel("Transactions")
plt.tight_layout()
plt.show()


## 4. Train/Test Split and Preprocessing

In [ ]:
# Remove the proxy score because it was used to create the target.
# It would be direct leakage if supplied to the classifier.
X = df.drop(columns=["FraudFlag", "FraudScore_Proxy"], errors="ignore")
y = df["FraudFlag"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training set:", X_train.shape, " | Test set:", X_test.shape)
print("Training positives:", y_train.sum(), " | Test positives:", y_test.sum())

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SkPipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            SkPipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]),
            categorical_features
        )
    ]
)


## 5. Baseline Models with SMOTE

SMOTE is placed **inside the imbalanced-learn pipeline**, after the train/test split. This means synthetic minority samples are generated only from the training data and cannot leak information from the test set.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba)
    })
    fitted_models[name] = pipe

baseline_results = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
display(baseline_results.round(4))


In [ ]:
# Detailed reports for both required algorithms.
for name, pipe in fitted_models.items():
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    print("=" * 70)
    print(name)
    print(classification_report(
        y_test, pred,
        target_names=["Legitimate", "Fraud Proxy"],
        zero_division=0
    ))
    print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
    print(f"PR-AUC:  {average_precision_score(y_test, proba):.4f}")


## 6. Random Forest Hyperparameter Tuning

In [ ]:
rf_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)
print(f"Best 5-fold CV ROC-AUC: {grid.best_score_:.4f}")


## 7. Tuned Model Evaluation

In [ ]:
best_model = grid.best_estimator_

final_pred = best_model.predict(X_test)
final_proba = best_model.predict_proba(X_test)[:, 1]

final_metrics = pd.DataFrame([{
    "Precision": precision_score(y_test, final_pred, zero_division=0),
    "Recall": recall_score(y_test, final_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, final_proba),
    "PR-AUC": average_precision_score(y_test, final_proba)
}])

display(final_metrics.round(4))

print("Classification report:")
print(classification_report(
    y_test, final_pred,
    target_names=["Legitimate", "Fraud Proxy"],
    zero_division=0
))

print("Confusion matrix:")
cm = confusion_matrix(y_test, final_pred)
display(pd.DataFrame(
    cm,
    index=["Actual Legitimate", "Actual Fraud Proxy"],
    columns=["Predicted Legitimate", "Predicted Fraud Proxy"]
))


In [ ]:
# Visual evaluation: confusion matrix and ROC curve
ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred,
    display_labels=["Legitimate", "Fraud Proxy"]
)
plt.title("Tuned Random Forest — Confusion Matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, final_proba)
plt.title("Tuned Random Forest — ROC Curve")
plt.tight_layout()
plt.show()


## 8. Feature Importance

In [ ]:
# Extract transformed feature names and Random Forest importances.
fitted_preprocessor = best_model.named_steps["preprocessor"]
rf = best_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importance = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

display(importance.head(15).to_frame("importance").round(4))

top = importance.head(15).sort_values()
plt.figure(figsize=(8, 6))
plt.barh(top.index, top.values)
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 9. Save the Final Model

The trained pipeline contains preprocessing, one-hot encoding, SMOTE, and the tuned Random Forest, so the same transformations can be reused when predicting new transactions.


In [ ]:
MODEL_PATH = "models/fraud_detection_random_forest_smote.joblib"
joblib.dump(best_model, MODEL_PATH)

print(f"Saved model to: {MODEL_PATH}")


## 10. Final Findings

1. The dataset is highly imbalanced after creating the proxy target, so Accuracy is not treated as the primary metric.
2. SMOTE is applied only to the training portion of the data.
3. Logistic Regression provides a strong linear baseline.
4. Random Forest captures nonlinear relationships and is tuned using 5-fold ROC-AUC.
5. The final model is evaluated using Precision, Recall, ROC-AUC and PR-AUC.
6. **Most important limitation:** the target is a proxy label, not verified fraud ground truth. For a real fraud-detection deployment, a genuine historical fraud label is required.
